In [10]:
import os
import io
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import httpx
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.2f}".format)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
(DATA_DIR / "ct_boundaries").mkdir(exist_ok=True)
(DATA_DIR / "census_profile").mkdir(exist_ok=True)

(DATA_DIR / "cimd").mkdir(exist_ok=True)

ARCGIS_TOKEN = os.getenv("ARCGIS_TOKEN", "")
if not ARCGIS_TOKEN:
    print("⚠️  ARCGIS_TOKEN not set — B1/B2 Esri sources will be skipped gracefully")
else:
    print("✅ ARCGIS_TOKEN loaded")

print("Setup complete. DATA_DIR:", DATA_DIR.resolve())

ModuleNotFoundError: No module named 'numpy'

In [ ]:
# A1 — StatsCan Census Tract boundaries 2021
# Cartographic boundary file (simplified), national coverage, ~15 MB zip
CT_URL = (
    "https://www12.statcan.gc.ca/census-recensement/2021/geo/sip-pis/"
    "boundary-limites/files-fichiers/lct_000b21a_e.zip"
)
CT_ZIP = DATA_DIR / "ct_boundaries" / "lct_000b21a_e.zip"

if not CT_ZIP.exists():
    print("Downloading CT boundaries (~15 MB)...")
    with httpx.Client(follow_redirects=True, timeout=120) as client:
        r = client.get(CT_URL)
        r.raise_for_status()
    CT_ZIP.write_bytes(r.content)
    print(f"Saved to {CT_ZIP}")
else:
    print(f"Using cached {CT_ZIP}")

with zipfile.ZipFile(CT_ZIP) as z:
    z.extractall(DATA_DIR / "ct_boundaries")

# Read — geopandas can read the shapefile directly from the extracted dir
shp_files = list((DATA_DIR / "ct_boundaries").glob("*.shp"))
assert shp_files, "No .shp file found after extraction"
gdf_ct = gpd.read_file(shp_files[0])

print("Raw columns:", gdf_ct.columns.tolist())
print("Raw CRS:", gdf_ct.crs)
print("Raw shape:", gdf_ct.shape)

# Filter to CMAs 35535 (Toronto CMA — covers Mississauga + Brampton) and 35537 (Hamilton)
# CMAUID column name may be 'CMAUID' or 'CMAPUID' — check print above and adjust
# Detect the CMA UID column name (varies between StatsCan releases)
cma_col = "CMAUID" if "CMAUID" in gdf_ct.columns else "CMAPUID"
if cma_col not in gdf_ct.columns:
    raise KeyError(f"Neither CMAUID nor CMAPUID found. Columns: {gdf_ct.columns.tolist()}")
gdf_ct = gdf_ct[gdf_ct[cma_col].astype(str).isin(["35535", "35537"])].copy()
gdf_ct = gdf_ct.to_crs("EPSG:4326").reset_index(drop=True)

print(f"\nFiltered to {len(gdf_ct)} CTs in CMAs 35535 + 35537")
gdf_ct[["CTUID", "CTNAME", cma_col, "CMANAME"]].head(3)


In [ ]:
# ASSERTION — run after the fetch cell to verify
assert "gdf_ct" in dir(), "gdf_ct not defined — run the fetch cell"
assert len(gdf_ct) >= 350, f"Expected ≥350 CTs in scope, got {len(gdf_ct)}"
assert gdf_ct.crs.to_epsg() == 4326, f"Expected EPSG:4326, got {gdf_ct.crs}"
assert "CTUID" in gdf_ct.columns, "CTUID column missing"
assert gdf_ct.geometry.notnull().all(), "Null geometries found"
print(f"✅ A1 assertions pass — {len(gdf_ct)} CTs, CRS: {gdf_ct.crs}")


In [ ]:
# A2 — Census Profile 2021, CT level (English, national)
# Large file (~1.5 GB unzipped). Downloads the zip (~200 MB), extracts, reshapes.
CENSUS_URL = (
    "https://www12.statcan.gc.ca/census-recensement/2021/dp-pd/prof/details/"
    "download-telecharger/comp/GetFile.cfm?Lang=E&FILETYPE=CSV&GEONO=044"
)
CENSUS_ZIP = DATA_DIR / "census_profile" / "98-401-X2021044_English_CSV.zip"
CENSUS_CSV = DATA_DIR / "census_profile" / "98-401-X2021044_English_CSV_data.csv"

if not CENSUS_CSV.exists():
    if not CENSUS_ZIP.exists():
        print("Downloading Census Profile CSV (~200 MB)...")
        with httpx.Client(follow_redirects=True, timeout=300) as client:
            r = client.get(CENSUS_URL)
            r.raise_for_status()
        CENSUS_ZIP.write_bytes(r.content)
        print("Download complete")
    print("Extracting...")
    with zipfile.ZipFile(CENSUS_ZIP) as z:
        z.extractall(DATA_DIR / "census_profile")
    print("Extracted")
else:
    print(f"Using cached {CENSUS_CSV}")

# Read only the rows we need to avoid loading 1.5 GB into memory
# The CSV is long-format: one row per (GEO_CODE, CHARACTERISTIC_ID)
# We filter to the 8 characteristic IDs we need
TARGET_CHARS = {1, 133, 1680, 1681, 1682, 1820, 1838, 1839}
CHAR_LABELS  = {
    1:    "population",
    133:  "median_income",
    1682: "renter_count",
    1681: "owner_count",
    1820: "total_period_construct",
    1838: "dwell_pre1960",
    1839: "dwell_1961_1980",
}

chunks = []
for chunk in pd.read_csv(CENSUS_CSV, chunksize=50_000, low_memory=False, encoding="latin-1"):
    chunk.columns = chunk.columns.str.strip()
    mask = chunk["CHARACTERISTIC_ID"].isin(TARGET_CHARS)
    chunks.append(chunk[mask])

df_long = pd.concat(chunks, ignore_index=True)
print(f"Filtered rows: {len(df_long)}")
print("Columns:", df_long.columns.tolist()[:10])

# Validate expected column names before pivoting (FIX 1)
geo_col = "ALT_GEO_CODE"
val_col = "C1_COUNT_TOTAL"
if geo_col not in df_long.columns:
    # Try alternate geo column names
    for candidate in ["GEO_CODE (POR)", "GEO_CODE", "DGUID"]:
        if candidate in df_long.columns:
            geo_col = candidate
            break
    else:
        raise KeyError(f"No geo column found. Columns: {df_long.columns.tolist()}")
if val_col not in df_long.columns:
    for candidate in ["C1_COUNT_TOTAL", "C1_COUNT_TOTAL:", "DATA_QUALITY_FLAG"]:
        if candidate in df_long.columns:
            val_col = candidate
            break
    else:
        raise KeyError(f"No value column found. Columns: {df_long.columns.tolist()}")
print(f"Using geo_col='{geo_col}', val_col='{val_col}'")

df_wide = (
    df_long[df_long["CHARACTERISTIC_ID"].isin(CHAR_LABELS)]
    .pivot_table(index=geo_col, columns="CHARACTERISTIC_ID", values=val_col, aggfunc="first")
    .reset_index()
    .rename(columns={**{k: v for k, v in CHAR_LABELS.items()}, geo_col: "CTUID"})
)

# Post-pivot source column validation (FIX 2)
expected_wide_cols = ["owner_count", "renter_count", "dwell_pre1960", "dwell_1961_1980", "total_period_construct"]
missing_wide = [c for c in expected_wide_cols if c not in df_wide.columns]
if missing_wide:
    print(f"⚠️  Missing derived columns (characteristic IDs may differ): {missing_wide}")
    print(f"   Available columns: {df_wide.columns.tolist()}")

df_wide["CTUID"] = df_wide["CTUID"].astype(str).str.zfill(7)

# Derived columns
df_wide["total_dwellings"] = df_wide.get("owner_count", pd.Series(0)) + df_wide.get("renter_count", pd.Series(0))
df_wide["pct_renters"]  = df_wide["renter_count"]  / df_wide["total_dwellings"].replace(0, np.nan)
df_wide["pct_pre1980"]  = (
    (df_wide.get("dwell_pre1960", pd.Series(0)).fillna(0) + df_wide.get("dwell_1961_1980", pd.Series(0)).fillna(0))
    / df_wide["total_period_construct"].replace(0, np.nan)
)
df_wide["median_income"] = pd.to_numeric(df_wide.get("median_income", pd.Series(dtype=float)), errors="coerce")

df_census = df_wide[["CTUID","population","median_income","pct_renters","pct_pre1980"]].copy()
print(f"\ndf_census shape: {df_census.shape}")
df_census.head(3)

In [ ]:
# ASSERTION
assert "df_census" in dir(), "df_census not defined"
assert {"CTUID", "population", "median_income", "pct_renters", "pct_pre1980"}.issubset(df_census.columns),     f"Missing columns. Got: {df_census.columns.tolist()}"
assert len(df_census) >= 350, f"Expected ≥350 rows, got {len(df_census)}"
null_pct = df_census[["median_income", "pct_renters", "pct_pre1980"]].isnull().mean()
assert (null_pct < 0.10).all(), f"High null rate in core columns:\n{null_pct}"
print(f"✅ A2 assertions pass — {len(df_census)} CTs")
print(df_census[["CTUID","median_income","pct_renters","pct_pre1980"]].describe())